# Embedding Images and Text Together [Step 1 - CLIP and Shared Embedding Spaces]

> **MLCourse - Agentic AI - Multi-Modal RAG**

### What you will learn

1. How CLIP maps images and text into a single shared embedding space.
2. Why OpenAI has no CLIP-style image embedding, and the caption-then-embed
   pipeline you use instead.
3. Computing cross-modal similarity: find text that matches an image and vice versa.
4. Why shared embedding spaces are the foundation of multi-modal RAG.

In traditional RAG, we embed text chunks and query with text. Multi-modal RAG
extends this: embed images AND text so a text query can retrieve relevant
images, and an image query can retrieve relevant text. The bridge is a SHARED
embedding space where both modalities live as vectors.

In [1]:
# ---------------------------------------------------------------------------
# Setup cell (identical in every MLCourse notebook): imports, TRACK walker,
# DATA folder creation, .env loading, matplotlib inline magic - guarded so
# the file also runs as a plain script outside Jupyter.
# ---------------------------------------------------------------------------
from pathlib import Path


def find_track(start: Path, target: str = "03_agentic_ai") -> Path:
    """Climb parent folders until a directory named ``target`` shows up."""
    for candidate in [start, *start.parents]:
        probe = candidate / target
        if probe.is_dir():
            return probe
    raise FileNotFoundError(
        f"Could not find '{target}' above {start}. "
        "Open this notebook from inside the MLCourse repository."
    )


TRACK = find_track(Path.cwd())       # .../MLCourse/03_agentic_ai
DATA = TRACK / "data"                # one shared data folder for the track
DATA.mkdir(exist_ok=True)            # no-op when it already exists

from dotenv import load_dotenv       # noqa: E402  reads KEY=value files

load_dotenv()                        # .env beside the current directory
load_dotenv(TRACK / ".env")          # .env at the track root

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass                             # silently skip the magic outside IPython

import matplotlib.pyplot as plt      # noqa: E402

print("[setup] TRACK:", TRACK)
print("[setup] DATA :", DATA)

[setup] TRACK: D:\projects\python\MLCourse\03_agentic_ai
[setup] DATA : D:\projects\python\MLCourse\03_agentic_ai\data


### Part 1: Why CLIP Changed Everything


### Install open_clip (the open-source CLIP implementation) if needed.


In [ ]:
import subprocess, sys

try:
    import open_clip                       # open-source CLIP models
    print("[OK] open_clip already installed")
except ImportError:
    print("[INSTALL] open_clip not found -- installing...")
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           "open-clip-torch", "torchvision", "-q"])
    import open_clip


### Part 2: Load a CLIP Model Locally


In [ ]:
#
# We use ViT-B-32 (Vision Transformer, Base, 32x32 patches) with the
# laion2b_s34b_b79k pretrained weights. This runs 100% locally after
# the one-time model download (~350 MB).

import torch
from PIL import Image

# Choose device: CUDA if available, then MPS (Apple Silicon), else CPU.
device = "cuda" if torch.cuda.is_available() else (
    "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"[CLIP] Using device: {device}")

# Load CLIP model, preprocessing transform, and tokenizer together.
model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k"
)
tokenizer = open_clip.get_tokenizer("ViT-B-32")

model = model.to(device).eval()           # move to device, set eval mode
print("[CLIP] Model loaded: ViT-B-32 (laion2b_s34b_b79k)")
print(f"[CLIP] Embedding dimension: {model.visual.output_dim}")


### Part 3: Embed Text with CLIP


In [ ]:
#
# CLIP tokenizes text and passes it through a transformer to produce
# a fixed-size embedding vector. All text vectors are L2-normalized
# (unit length) so cosine similarity equals dot product.

SAMPLE_TEXTS = [
    "a diagram showing neural network architecture",
    "a transformer attention mechanism illustration",
    "a bar chart comparing model performance",
    "attention is all you need",
    "multi-head self-attention mechanism",
    "a photo of a cat sitting on a windowsill",
    "the sky is blue on a clear day",
]

# Tokenize all texts in one batch (faster than one at a time).
text_tokens = tokenizer(SAMPLE_TEXTS).to(device)

with torch.no_grad():                     # no gradients needed for inference
    text_features = model.encode_text(text_tokens)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)  # L2-normalize

print(f"[TEXT] Embedded {len(SAMPLE_TEXTS)} texts -> shape {text_features.shape}")
print(f"[TEXT] Each vector has {text_features.shape[1]} dimensions")


### Part 4: Embed Images with CLIP


In [ ]:
#
# The same model embeds images through its visual encoder. After
# L2-normalization, image vectors live in the SAME space as text vectors.
# We use the three sample images from our data folder.

IMAGES_DIR = DATA / "images"
sample_images = ["architecture_diagram.png", "bar_chart.png", "flowchart.png"]

image_tensors = []
image_names = []
for name in sample_images:
    img_path = IMAGES_DIR / name
    if img_path.exists():
        img = Image.open(img_path).convert("RGB")
        image_tensors.append(preprocess(img).unsqueeze(0))
        image_names.append(name)
        print(f"[IMG] Loaded {name}: {img.size[0]}x{img.size[1]}")
    else:
        print(f"[SKIP] {name} not found")

if not image_tensors:
    raise FileNotFoundError(
        f"No images found in {IMAGES_DIR}. "
        "Make sure sample images exist."
    )

# Stack all image tensors into a single batch and move to device.
image_batch = torch.cat(image_tensors, dim=0).to(device)

with torch.no_grad():
    image_features = model.encode_image(image_batch)
    image_features = image_features / image_features.norm(dim=-1, keepdim=True)

print(f"\n[IMG] Embedded {len(image_names)} images -> shape {image_features.shape}")
print(f"[IMG] Each vector has {image_features.shape[1]} dimensions")
print(f"[IMG] Text and image vectors share dimensionality: "
      f"{text_features.shape[1] == image_features.shape[1]}")


### Part 5: Cross-Modal Similarity Matrix


In [ ]:
#
# Now the magic: compute cosine similarity between EVERY text and EVERY
# image. High values mean the text describes the image well.
# Since vectors are L2-normalized, we just use matrix multiply.

sim_matrix = (text_features @ image_features.T).cpu().numpy()

print("Cross-modal similarity matrix (text rows x image columns):")
print("=" * 70)

# Header row with image names.
header = f"{'Text':<45}" + "".join(f"{n:>12}" for n in image_names)
print(header)
print("-" * 70)

# Each row: text snippet + similarity scores.
for i, text in enumerate(SAMPLE_TEXTS):
    short = text[:43] + ".." if len(text) > 45 else text
    scores = " ".join(f"{sim_matrix[i, j]:>12.4f}" for j in range(len(image_names)))
    print(f"{short:<45}{scores}")


### Part 6: Visualize the Similarity Heatmap


In [ ]:
#
# A heatmap makes the cross-modal alignment obvious. Bright cells mean
# high similarity (the text matches the image). We expect technical
# diagram texts to score high with the architecture/flowchart images.

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sim_matrix, cmap="YlOrRd", aspect="auto", vmin=0, vmax=1)

# Labels on axes.
ax.set_xticks(range(len(image_names)))
ax.set_xticklabels(image_names, rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(len(SAMPLE_TEXTS)))
ax.set_yticklabels(SAMPLE_TEXTS, fontsize=9)

# Annotate each cell with its numeric value.
for i in range(len(SAMPLE_TEXTS)):
    for j in range(len(image_names)):
        val = sim_matrix[i, j]
        color = "white" if val > 0.6 else "black"
        ax.text(j, i, f"{val:.3f}", ha="center", va="center",
                fontsize=8, color=color)

plt.colorbar(im, label="Cosine Similarity", shrink=0.8)
plt.title("CLIP Cross-Modal Similarity: Text vs Images", fontsize=12)
plt.tight_layout()
plt.show()


### Part 7: Text-to-Image Retrieval


In [ ]:
#
# Given a text query, find the most similar image. This is the foundation
# of text-to-image search in multi-modal RAG.

def retrieve_images(query_text: str, top_k: int = 3) -> list[tuple[str, float]]:
    """Find the top_k most similar images for a text query using CLIP."""
    tokens = tokenizer([query_text]).to(device)
    with torch.no_grad():
        query_vec = model.encode_text(tokens)
        query_vec = query_vec / query_vec.norm(dim=-1, keepdim=True)

    # Dot product = cosine similarity (vectors are L2-normalized).
    sims = (query_vec @ image_features.T).cpu().numpy().flatten()
    ranked = sorted(zip(image_names, sims), key=lambda x: -x[1])
    return ranked[:top_k]

# Try several queries.
QUERIES = [
    "an attention mechanism diagram",
    "a bar chart with numbers",
    "neural network layers",
]

for q in QUERIES:
    print(f"\nQuery: \"{q}\"")
    print("-" * 50)
    for rank, (img_name, score) in enumerate(retrieve_images(q), 1):
        bar = "#" * int(score * 40)
        print(f"  #{rank} {score:.4f}  {img_name:<30} {bar}")


### Part 8: Image-to-Text Retrieval


In [ ]:
#
# The reverse direction: given an image, find the most relevant text.
# This is useful when you have an image and want to find related
# document chunks or captions.

def retrieve_texts(query_image_path: str, top_k: int = 3) -> list[tuple[str, float]]:
    """Find the top_k most similar texts for an image query using CLIP."""
    img = Image.open(query_image_path).convert("RGB")
    img_tensor = preprocess(img).unsqueeze(0).to(device)
    with torch.no_grad():
        query_vec = model.encode_image(img_tensor)
        query_vec = query_vec / query_vec.norm(dim=-1, keepdim=True)

    sims = (query_vec @ text_features.T).cpu().numpy().flatten()
    ranked = sorted(zip(SAMPLE_TEXTS, sims), key=lambda x: -x[1])
    return ranked[:top_k]

# Use the first available sample image as query.
if image_tensors:
    query_img = str(DATA / "images" / image_names[0])
    print(f"\nImage query: {image_names[0]}")
    print("-" * 50)
    for rank, (txt, score) in enumerate(retrieve_texts(query_img), 1):
        print(f"  #{rank} {score:.4f}  {txt}")


### Part 9: OpenAI Vision Embeddings (Paid, Guarded)


In [ ]:
#
# IMPORTANT CORRECTION TO A COMMON MISCONCEPTION: OpenAI does NOT offer a
# multimodal embedding endpoint. `client.embeddings.create` accepts TEXT only,
# and passing `model="gpt-4o"` to it returns
#   403 "You are not allowed to generate embeddings from this model".
# There is no OpenAI equivalent of CLIP's shared image/text vector space.
#
# The real OpenAI recipe for cross-modal retrieval is TWO steps:
#   1. CAPTION the image with a vision chat model (gpt-4o-mini).
#   2. EMBED that caption with a text embedding model (text-embedding-3-small).
# Everything downstream is then ordinary text retrieval.
#
# Contrast with CLIP above:
#   CLIP    - one shared space, image and text embedded directly, no LLM call.
#   Caption - text-only space; the image is first translated into words, so
#             retrieval quality is capped by how good the caption is, and any
#             detail the caption omits is invisible to search.
#
# This section is GUARDED -- it only runs when an OPENAI_API_KEY is present.

import os

OPENAI_KEY = os.environ.get("OPENAI_API_KEY", "")

if not OPENAI_KEY:
    print("[GUARD] No OPENAI_API_KEY found -- skipping OpenAI vision section.")
    print("[GUARD] The CLIP sections above demonstrate the same concepts.")
else:
    import base64
    import numpy as np
    from openai import OpenAI

    client = OpenAI()
    VISION_MODEL = "gpt-4o-mini"
    EMBED_MODEL = "text-embedding-3-small"

    def caption_image(img_path: str) -> str:
        """Step 1: turn an image into a retrieval-friendly sentence."""
        with open(img_path, "rb") as f:
            b64 = base64.b64encode(f.read()).decode()
        resp = client.chat.completions.create(
            model=VISION_MODEL,
            messages=[{"role": "user", "content": [
                {"type": "text",
                 "text": "Describe this image in one sentence, for search indexing."},
                {"type": "image_url",
                 "image_url": {"url": f"data:image/png;base64,{b64}"}},
            ]}],
            max_tokens=60,
        )
        return resp.choices[0].message.content.strip()

    def embed_texts(texts: list[str]) -> np.ndarray:
        """Step 2: embed text (the ONLY thing the embeddings API accepts)."""
        resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
        return np.array([d.embedding for d in resp.data])

    test_img = str(DATA / "images" / image_names[0])
    caption = caption_image(test_img)
    print(f"[openai] image      : {image_names[0]}")
    print(f"[openai] caption    : {caption}")

    # Embed the caption and the candidate texts in ONE shared text space.
    vecs = embed_texts([caption] + SAMPLE_TEXTS)
    cap_vec, text_vecs = vecs[0], vecs[1:]
    oai_sims = (text_vecs @ cap_vec) / (
        np.linalg.norm(text_vecs, axis=1) * np.linalg.norm(cap_vec)
    )

    print(f"[openai] embedding  : {EMBED_MODEL}, {vecs.shape[1]} dims")
    print()
    print("Caption-vs-text similarity ranking:")
    for rank, idx in enumerate(np.argsort(-oai_sims), 1):
        print(f"  #{rank} {oai_sims[idx]:.4f}  {SAMPLE_TEXTS[idx]}")
    print()
    print("Compare the ordering with the CLIP result above: CLIP scores the")
    print("image directly, this pipeline scores a sentence ABOUT the image.")


### Summary


In [ ]:
#
# 1. CLIP produces vectors for images and text in a SHARED space (512 dims).
# 2. Cosine similarity across modalities tells us "how well does this text
#    describe this image?"
# 3. Text-to-image and image-to-text retrieval use the same math: dot product.
# 4. OpenAI gpt-4o can also embed both modalities (requires API key).
# 5. These shared spaces are the foundation of multi-modal RAG indexing
#    and retrieval, which we build in notebooks 02-04.

print("\n[COMPLETE] Module 21 Notebook 1: Image-Text Embeddings")
print("  - CLIP shared embedding space demonstrated")
print("  - Cross-modal similarity matrix computed")
print("  - Text-to-image retrieval working")
print("  - Image-to-text retrieval working")
